# Multimodal Messages

In [1]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [2]:
print(uploader.value)

({'name': '1793098025.png', 'type': 'image/png', 'size': 3894723, 'content': <memory at 0x7a6e547d5600>, 'last_modified': datetime.datetime(2026, 10, 27, 10, 53, 13, 447000, tzinfo=datetime.timezone.utc)},)


In [3]:
import base64

uploaded_file = uploader.value[0]

content_mv = uploaded_file['content']

img_bytes = bytes(content_mv)

img_base64 = base64.b64encode(img_bytes).decode('utf-8')

In [4]:

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

load_dotenv()

True

In [5]:
groq_api_key = os.getenv("GROQ_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [6]:
model = ChatGroq(
    api_key=groq_api_key,
    model="qwen/qwen3.8-27b",
    temperature=0.7,
    max_retries=3,
    max_tokens=1024,
    timeout=30
)

tavily_search = TavilySearch(
    tavily_api_key=tavily_api_key,
    max_results=5,
    topic="general",
    search_depth="basic"
)

In [7]:
from pprint import pprint
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

saver = InMemorySaver()

agent = create_agent(
    model=model,
    system_prompt="You are a helpful assistant that can answer questions and provide information.",
    tools=[tavily_search],
    checkpointer=saver
)

In [10]:
image_url = f"data:image/png;base64,{img_base64}"

multimodal_question = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "What is the content of this image?"
        },
        {
            "type": "image_url",
            "image_url": {
                "url": image_url
            }
        }
    ]
)


In [11]:
response = agent.invoke(
    {"messages": [multimodal_question]},
    config={
        "configurable": {
            "thread_id": "thread_1"
        }
    }
)

pprint(response["messages"][-1].content)

('The image displays a collage of four different views of the same laptop, '
 'showcasing its design and features.\n'
 '\n'
 'Here is a detailed breakdown of the content in each panel:\n'
 '\n'
 '- **Top-Left Panel:**\n'
 '    - This is a three-quarter view of the open laptop.\n'
 '    - It has a silver or light gray aluminum body.\n'
 '    - The screen is on, displaying a dark wallpaper with a logo in the '
 'center. The time "10:09" is visible in the top-right corner of the screen.\n'
 '    - The keyboard is black with a standard layout.\n'
 '    - A large trackpad is visible below the keyboard.\n'
 '    - On the right side of the laptop, several ports are visible, including '
 'what appear to be USB-C ports, a headphone jack, and possibly an SD card '
 'slot.\n'
 '\n'
 '- **Bottom-Left Panel:**\n'
 "    - This is a close-up shot of the top bezel of the laptop's screen.\n"
 '    - It clearly shows the camera (webcam) located in the center of the top '
 'bezel.\n'
 '    - The same log